# 29. Export and import a B+/B- `CPRealImag` model pair

**Objectives:**
- Build a B+/B- `DecayModel` pair sharing one `CPRealImag` coefficient via
  `.for_charge(+1)`/`.for_charge(-1)`, with `x`/`y`/`dx`/`dy` built from
  `Parameter.coefficient(...)` so the shared identity is meaningful.
- Export the pair with `export_cp_models` and reload it with `import_cp_models`.
- Confirm the shared coefficient's `Parameter` objects come back as the exact same
  object (`is`, not just `==`) in both reloaded models, per `docs/model_io.md`.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

from dalitzplotfitter import (
    CPRealImag, DecayChannel, DecayModel, NonResonant, Parameter, RealImag,
    Resonance, export_cp_models, import_cp_models,
)

## 1. Build the B+/B- pair from a shared `CPRealImag`

`c_q = (x + q*dx) + i*(y + q*dy)`, `q = +1` or `-1`. Building `x`, `y`, `dx`, `dy` as
`Parameter.coefficient(...)` (rather than bare floats) is what makes the coefficient
*floatable* and what makes "the same object in both models" a meaningful, checkable
claim below -- a plain float would just compare equal, not be identical.

In [2]:
cp = CPRealImag(*[
    Parameter.coefficient(f"NR.{name}", value, owner="NR", bounds=bounds)
    for name, value, bounds in [
        ("x", 0.6, (-2, 2)), ("y", 0.2, (-2, 2)),
        ("dx", 0.05, (-0.4, 0.4)), ("dy", -0.03, (-0.4, 0.4)),
    ]
])

def make_model(parent, daughters, charge):
    return DecayModel(
        DecayChannel(parent, daughters),
        [Resonance("Kstar", pair=(0, 2), coefficient=RealImag(1.0, 0.0),
                   mass=0.8958, width=0.0474, spin=1),
         NonResonant(cp.for_charge(charge), name="NR")],
        normalization_method="square-dalitz", normalization_pair=(0, 2),
        normalization_resolution=40,
    )

plus_model = make_model("B+", ("K+", "pi+", "pi-"), +1)
minus_model = make_model("B-", ("K-", "pi-", "pi+"), -1)

## 2. Export, reload, and check shared identity

`export_cp_models`/`import_cp_models` encode and decode the two models through one
shared parameter registry, so a coefficient built with `CPRealImag.for_charge` -- shared
between the B+ and B- models -- decodes back to the exact same `Parameter` object in
both, not just an equal one.

In [3]:
export_path = "tutorial_29_cp_models.json"
export_cp_models(plus_model, export_path, minus_model)

restored_plus, restored_minus = import_cp_models(export_path)

plus_nr = next(c for c in restored_plus.components if c.name == "NR")
minus_nr = next(c for c in restored_minus.components if c.name == "NR")

for attr in ("x", "y", "dx", "dy"):
    plus_param = getattr(plus_nr.coefficient, attr)
    minus_param = getattr(minus_nr.coefficient, attr)
    assert plus_param is minus_param, f"{attr} should be the identical shared Parameter object"
    print(f"{attr}: shared Parameter object -> {plus_param.name} = {plus_param.value}")

assert plus_nr.coefficient.charge == +1
assert minus_nr.coefficient.charge == -1
print("Shared CPRealImag parameters preserved object identity across both reloaded models.")

x: shared Parameter object -> NR.x = 0.6
y: shared Parameter object -> NR.y = 0.2
dx: shared Parameter object -> NR.dx = 0.05
dy: shared Parameter object -> NR.dy = -0.03
Shared CPRealImag parameters preserved object identity across both reloaded models.


This is what makes a reloaded `CPJointNLL`/`CPFitSession` behave correctly: the joint
normalization threads one shared `Parameter` list through both charge models (see
`docs/cp_coefficients.md`, "CP fits share one normalization across charges"), and that
only works if `dx`/`dy` really are the *same* fittable object on both sides, not two
independently-floating copies that merely start out equal.

## Continue learning

See [docs/model_io.md](../../docs/model_io.md) and
[docs/cp_coefficients.md](../../docs/cp_coefficients.md). Return to
[the course guide](TUTORIALS.md).